# DentPanX-AI — Revision 2 reproducibility

Run from top to bottom in Google Colab with a T4 GPU. Default mode reuses archived full-precision inputs; it does not retrain or rerun inference. Original results are preserved. This notebook reproduces AP bootstrap intervals and the six fixed examples in Supplementary Figure S1.

The public source dataset, model checkpoint, split CSV and archived NPZ/CSV files must be supplied separately. Edit paths below for another installation. No patient-level independence or independent clinical adjudication is implied.

## 1. Mount Drive and verify checkpoint / split

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import json
import pandas as pd

MY_DRIVE = Path("/content/drive/MyDrive")

# Ekran görüntüsündeki yeni ad ve eski kodlardaki ad
candidates = [
    MY_DRIVE / "TT-Makale-MOPG_AI_Outputs_Clean7",
    MY_DRIVE / "MOPG_AI_Outputs_Clean7",
]

found = [p for p in candidates if p.is_dir()]

print("Bulunan proje klasörleri:")
for p in found:
    print(" -", p)

if len(found) != 1:
    raise RuntimeError(
        "Tek bir proje klasörü belirlenemedi. "
        "Çıktıyı paylaşın; doğru klasörü birlikte seçelim."
    )

PROJECT_ROOT = found[0]

MODEL_PATH = PROJECT_ROOT / "final_model" / "best.pt"
METADATA_PATH = PROJECT_ROOT / "final_model" / "final_model_metadata.json"
SPLIT_PATH = PROJECT_ROOT / "split_reports" / "split_assignments.csv"

print("\nDosya kontrolü:")
for p in [MODEL_PATH, METADATA_PATH, SPLIT_PATH]:
    print(f"{'VAR' if p.is_file() else 'EKSİK'} | {p}")
    if not p.is_file():
        raise FileNotFoundError(p)

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

model_hash = sha256_file(MODEL_PATH)
expected_hash = (
    "6e32bba3e966a0cf3017eed689403953"
    "71f9ec9bf4813b0b13456a41c58d4861"
)

print("\nMODEL SHA-256:")
print(model_hash)
print("Önceki kayıtla eşleşiyor:", model_hash == expected_hash)

print("\nMODEL METADATA:")
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
print(json.dumps(metadata, indent=2, ensure_ascii=False))

split_df = pd.read_csv(SPLIT_PATH)

print("\nBÖLME DOSYASI SHA-256:")
print(sha256_file(SPLIT_PATH))

print("\nBÖLME DOSYASI:")
print("Satır sayısı:", len(split_df))
print("Sütunlar:", split_df.columns.tolist())
display(split_df.head())

for col in split_df.columns:
    if str(col).strip().lower() in {"split", "partition", "set"}:
        print(f"\n'{col}' dağılımı:")
        print(split_df[col].value_counts(dropna=False))

if model_hash != expected_hash:
    raise RuntimeError(
        "Model önceki kayıtla eşleşmiyor. Çıkarıma geçmeden kontrol edilmeli."
    )

print("\nİlk kontrol tamamlandı. Henüz çıkarım veya eğitim yapılmadı.")

## 2. Verify source archive and reconstruct the 315-image test set

In [ ]:
import zipfile
from collections import defaultdict, Counter
from pathlib import PurePosixPath

# Önceki çalışmada kullanılan arşivin kayıtlı konumu
ZIP_PATH = (
    MY_DRIVE
    / "MOPG-7 A Multi-Clinic Dental Panoramic Radiograph"
    / "MOPG-7 A Multi-Clinic Dental Panoramic Radiograph"
    / "MOPG-7.zip"
)

if not ZIP_PATH.is_file():
    raise FileNotFoundError(
        f"Arşiv bu konumda bulunamadı:\n{ZIP_PATH}\n"
        "MOPG-7.zip dosyasının güncel Drive konumunu paylaşın."
    )

assert split_df["image_name"].is_unique, "Tekrarlanan görüntü adı var."
assert len(split_df) == 2095

test_names = (
    split_df.loc[split_df["split"] == "test", "image_name"]
    .astype(str)
    .tolist()
)
assert len(test_names) == 315
assert len({Path(n).stem for n in test_names}) == 315

print("Kaynak arşiv:", ZIP_PATH)
print("Boyut (MB):", round(ZIP_PATH.stat().st_size / 1024**2, 2))

image_index = defaultdict(list)
label_index = defaultdict(list)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for info in z.infolist():
        if info.is_dir():
            continue

        member = PurePosixPath(info.filename)
        suffix = member.suffix.lower()

        if suffix in {".jpg", ".jpeg", ".bmp", ".png"}:
            image_index[member.name].append(info.filename)
        elif suffix == ".txt":
            label_index[member.stem].append(info.filename)

    problems = []
    for name in test_names:
        images = image_index.get(name, [])
        labels = label_index.get(Path(name).stem, [])

        if len(images) != 1 or len(labels) != 1:
            problems.append({
                "image_name": name,
                "image_matches": len(images),
                "label_matches": len(labels),
            })

    print("\nTest görüntüsü sayısı:", len(test_names))
    print("Eksik veya belirsiz eşleşme:", len(problems))

    if problems:
        display(pd.DataFrame(problems))
        raise RuntimeError("Eşleşmeler çözülmeden devam edilmeyecek.")

    raw_counts = Counter()
    malformed = []

    for name in test_names:
        label_member = label_index[Path(name).stem][0]
        content = z.read(label_member).decode("utf-8-sig")

        for line_no, line in enumerate(content.splitlines(), start=1):
            if not line.strip():
                continue

            parts = line.split()
            try:
                if len(parts) != 5:
                    raise ValueError("Beklenen sütun sayısı 5")
                class_value = float(parts[0])
                if not class_value.is_integer():
                    raise ValueError("Sınıf kimliği tam sayı değil")
                cls = int(class_value)
                raw_counts[cls] += 1
            except ValueError:
                malformed.append((name, line_no, line))

print("\nHam test anotasyonları — sınıf dağılımı:")
for cls, count in sorted(raw_counts.items()):
    print(f"Sınıf {cls}: {count}")

print("\nToplam anotasyon:", sum(raw_counts.values()))
print("Biçim sorunu olan satır:", len(malformed))

if malformed:
    print(malformed[:10])
    raise RuntimeError("Etiket biçimleri kontrol edilmeli.")

expected_counts = {
    0: 411,
    1: 282,
    2: 305,
    3: 225,
    4: 130,
    5: 132,
    6: 34,
}

official_counts = {cls: raw_counts[cls] for cls in range(7)}
print("\nResmî sınıfların toplamı:", sum(official_counts.values()))
print("Önceki sınıf destekleriyle eşleşiyor:",
      official_counts == expected_counts)

assert official_counts == expected_counts, (
    "Sınıf sayıları önceki kayıtlarla uyuşmuyor."
)

print("\nİkinci kontrol tamamlandı. Arşiv değiştirilmedi.")

In [ ]:
import tempfile
import math
from collections import Counter
from PIL import Image

# Her çalıştırmada yeni, ayrı bir geçici klasör oluşturur.
LOCAL_ROOT = Path(tempfile.mkdtemp(
    prefix="DentPanX_R2_test_",
    dir="/content"
))
IMAGE_DIR = LOCAL_ROOT / "images"
LABEL_DIR = LOCAL_ROOT / "labels"
IMAGE_DIR.mkdir()
LABEL_DIR.mkdir()

clean_counts = Counter()
removed_counts = Counter()
empty_labels = []
manifest = []

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for name in test_names:
        assert Path(name).name == name, f"Beklenmeyen dosya adı: {name}"

        image_member = image_index[name][0]
        label_member = label_index[Path(name).stem][0]

        # Gerçek dosya kopyası; sembolik bağlantı kullanılmaz.
        image_bytes = z.read(image_member)
        image_path = IMAGE_DIR / name
        image_path.write_bytes(image_bytes)

        with Image.open(image_path) as img:
            img.verify()
        with Image.open(image_path) as img:
            width, height = img.size

        retained = []
        raw_text = z.read(label_member).decode("utf-8-sig")

        for line_no, line in enumerate(raw_text.splitlines(), start=1):
            line = line.strip()
            if not line:
                continue

            parts = line.split()
            assert len(parts) == 5, (name, line_no, line)

            cls_value = float(parts[0])
            assert cls_value.is_integer(), (name, line_no, line)
            cls = int(cls_value)

            if cls not in range(7):
                removed_counts[cls] += 1
                continue

            coords = [float(v) for v in parts[1:]]
            assert all(math.isfinite(v) for v in coords), (
                name, line_no, "Sonlu olmayan koordinat"
            )
            assert all(0 <= v <= 1 for v in coords), (
                name, line_no, "Normalize koordinat aralık dışında"
            )
            assert coords[2] > 0 and coords[3] > 0, (
                name, line_no, "Geçersiz kutu boyutu"
            )

            retained.append(line)
            clean_counts[cls] += 1

        label_path = LABEL_DIR / f"{Path(name).stem}.txt"
        label_text = "\n".join(retained)
        label_path.write_text(
            label_text + ("\n" if retained else ""),
            encoding="utf-8"
        )

        if not retained:
            empty_labels.append(name)

        manifest.append({
            "image_name": name,
            "zip_image_member": image_member,
            "zip_label_member": label_member,
            "width": width,
            "height": height,
            "retained_boxes": len(retained),
            "image_sha256": hashlib.sha256(image_bytes).hexdigest(),
            "clean_label_sha256": sha256_file(label_path),
        })

# Bu YAML yalnızca değerlendirme içindir.
DATA_YAML = LOCAL_ROOT / "data.yaml"
data_config = {
    "path": str(LOCAL_ROOT),
    "train": "images",
    "val": "images",
    "test": "images",
    "nc": 7,
    "names": [
        "Missing", "Dental Crown", "Root Canal", "Caries",
        "Broken Down", "Wisdom Teeth", "Healthy"
    ],
}
# JSON, geçerli bir YAML alt kümesidir.
DATA_YAML.write_text(
    json.dumps(data_config, indent=2),
    encoding="utf-8"
)

pd.DataFrame(manifest).to_csv(
    LOCAL_ROOT / "test_manifest.csv", index=False
)

assert len(list(IMAGE_DIR.iterdir())) == 315
assert len(list(LABEL_DIR.glob("*.txt"))) == 315
assert dict(clean_counts) == expected_counts
assert sum(clean_counts.values()) == 1519
assert len(empty_labels) == 4
assert not IMAGE_DIR.is_symlink()
assert all(not p.is_symlink() for p in IMAGE_DIR.iterdir())

print("Yerel test klasörü:", LOCAL_ROOT)
print("Görüntü sayısı:", len(manifest))
print("Etiket dosyası sayısı:", len(list(LABEL_DIR.glob("*.txt"))))
print("Korunan anotasyon:", sum(clean_counts.values()))
print("Kapsam dışı çıkarılan satırlar:", dict(removed_counts))
print("Boş etiketli görüntüler:", empty_labels)
print("Sınıf sayıları:", dict(sorted(clean_counts.items())))
print("YAML:", DATA_YAML)
print("\nTEST KÜMESİ HAZIR — henüz model çalıştırılmadı.")

## 3. Check environment and install the recorded Ultralytics version

In [ ]:
import sys
import platform
import importlib.metadata as md
import torch

print("Python:", sys.version.split()[0])
print("İşletim sistemi:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA sürümü:", torch.version.cuda)
print("GPU kullanılabilir:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU belleği (GB):",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    )
else:
    print(
        "\nGPU bağlı değil. Çıkarım öncesinde Colab çalışma zamanı "
        "ayarlarından GPU seçin. Oturum yeniden başlarsa önceki "
        "hücreleri sırayla yeniden çalıştırın."
    )

print("\nPaket sürümleri:")
for package in [
    "ultralytics", "torchvision", "numpy",
    "scipy", "pandas", "Pillow"
]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: KURULU DEĞİL")

print("\nDosyalar hâlâ erişilebilir mi?")
for path in [MODEL_PATH, DATA_YAML, IMAGE_DIR, LABEL_DIR]:
    print(f"{path.exists()} | {path}")

print("\nOrtam kontrolü tamamlandı; model çalıştırılmadı.")

In [ ]:
import subprocess
import sys

# Kurulum başarısız olursa hücre durur.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "ultralytics==8.4.115"
    ],
    check=True
)

import ultralytics
import torch
import torchvision
from ultralytics import RTDETR

print("\nKURULUM KONTROLÜ")
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

assert ultralytics.__version__ == "8.4.115"
assert torch.cuda.is_available()

# GPU üzerinde küçük bir işlemle erişimi doğrula.
x = torch.ones(4, device="cuda")
assert x.sum().item() == 4
del x

print("\nKurulum ve GPU kontrolü başarılı.")
print("Henüz model yüklenmedi veya test çıkarımı yapılmadı.")

## 4. Configure archive and a separate output directory

In [ ]:
import uuid, json, shutil, platform
import numpy as np
import ultralytics
from datetime import datetime, timezone
ARCHIVE_ROOT = PROJECT_ROOT / "revision2_ap_bootstrap" / "20261103_101644_6d74f2"
ARCHIVE_CAPTURE = ARCHIVE_ROOT / "bootstrap_capture_437e69"
ARCHIVE_FIG = ARCHIVE_ROOT / "fp_fn_coordinate_capture_20261104_193812_ab1422"
RECAPTURE = False  # True: rerun test inference; never train.
RUN_ROOT = PROJECT_ROOT / "revision2_reproduction" / (datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6])
RUN_ROOT.mkdir(parents=True, exist_ok=False)
current_map50 = 0.6755318157587847
current_map5095 = 0.29389012321773966
model = RTDETR(str(MODEL_PATH))
SELECTION_PATH = ARCHIVE_FIG / "final_selected_examples.csv"
assert SELECTION_PATH.is_file(), SELECTION_PATH
(RUN_ROOT / "environment.json").write_text(json.dumps({
    "python": platform.python_version(), "ultralytics": ultralytics.__version__,
    "torch": torch.__version__, "torchvision": torchvision.__version__,
    "numpy": np.__version__, "cuda": torch.version.cuda,
    "model_sha256": sha256_file(MODEL_PATH), "recapture": RECAPTURE
}, indent=2))
print("New output directory:", RUN_ROOT)

## 5. Load archived AP inputs (or optionally capture them again)

In [ ]:
if RECAPTURE:
    from ultralytics.models.rtdetr.val import RTDETRValidator
    import numpy as np
    import pandas as pd
    import json
    import uuid
    
    image_stats = []
    
    class BootstrapCaptureValidator(RTDETRValidator):
        def init_metrics(self, model):
            super().init_metrics(model)
    
            original_update_stats = self.metrics.update_stats
    
            def capture_and_update(stat):
                # Her görüntüyü, tahmin olmasa bile, ayrı kaydet.
                image_stats.append({
                    "image_name": str(stat["im_name"]),
                    "tp": np.asarray(stat["tp"]).copy(),
                    "conf": np.asarray(stat["conf"]).copy(),
                    "pred_cls": np.asarray(stat["pred_cls"]).copy(),
                    "target_cls": np.asarray(stat["target_cls"]).copy(),
                })
                return original_update_stats(stat)
    
            self.metrics.update_stats = capture_and_update
    
    
    capture_name = "bootstrap_capture_" + uuid.uuid4().hex[:6]
    CAPTURE_DIR = RUN_ROOT / capture_name
    
    capture_metrics = model.val(
        validator=BootstrapCaptureValidator,
        data=str(DATA_YAML),
        split="test",
        imgsz=640,
        batch=2,
        conf=0.001,
        iou=0.70,
        max_det=300,
        device=0,
        workers=2,
        half=False,
        plots=False,
        save_txt=False,
        save_json=False,
        project=str(RUN_ROOT),
        name=capture_name,
        exist_ok=False,
        verbose=False,
    )
    
    # Görüntü ve anotasyon bütünlüğü
    captured_names = [s["image_name"] for s in image_stats]
    
    assert len(image_stats) == 315, (
        f"315 yerine {len(image_stats)} görüntü yakalandı."
    )
    assert len(set(captured_names)) == 315
    assert set(captured_names) == set(test_names)
    
    all_targets = np.concatenate([
        s["target_cls"] for s in image_stats
    ]).astype(int)
    
    support = np.bincount(all_targets, minlength=7)
    assert support.tolist() == [expected_counts[c] for c in range(7)]
    assert len(all_targets) == 1519
    
    for s in image_stats:
        n = len(s["conf"])
        assert s["tp"].shape == (n, 10), (
            s["image_name"], s["tp"].shape
        )
        assert len(s["pred_cls"]) == n
        assert np.isfinite(s["conf"]).all()
    
    # Pickle gerektirmeyen NPZ biçimiyle tam hassasiyetli kayıt
    CAPTURE_DIR.mkdir(parents=True, exist_ok=True)
    
    arrays = {
        "image_names": np.asarray(captured_names, dtype=str),
        "iou_thresholds": np.linspace(0.50, 0.95, 10),
    }
    
    for i, s in enumerate(image_stats):
        for key in ["tp", "conf", "pred_cls", "target_cls"]:
            arrays[f"{key}_{i:04d}"] = s[key]
    
    STATS_PATH = CAPTURE_DIR / "image_level_bootstrap_inputs.npz"
    np.savez_compressed(STATS_PATH, **arrays)
    
    capture_map50 = float(capture_metrics.box.map50)
    capture_map5095 = float(capture_metrics.box.map)
    
    capture_summary = {
        "images": len(image_stats),
        "annotations": len(all_targets),
        "annotations_per_class": support.tolist(),
        "images_without_reference_boxes": sum(
            len(s["target_cls"]) == 0 for s in image_stats
        ),
        "predictions": sum(len(s["conf"]) for s in image_stats),
        "map50": capture_map50,
        "map50_95": capture_map5095,
        "difference_from_previous_run_map50":
            capture_map50 - current_map50,
        "difference_from_previous_run_map50_95":
            capture_map5095 - current_map5095,
    }
    
    (CAPTURE_DIR / "capture_summary.json").write_text(
        json.dumps(capture_summary, indent=2),
        encoding="utf-8"
    )
    
    print("\nBOOTSTRAP GİRDİLERİ")
    print(json.dumps(capture_summary, indent=2))
    print("\nKaydedilen dosya:", STATS_PATH)
    print("Dosya boyutu (MB):", round(STATS_PATH.stat().st_size / 1024**2, 2))
    
    print("\nBir önceki değerlendirmeyle fark:")
    print(f"mAP50    : {capture_map50-current_map50:+.10f}")
    print(f"mAP50–95 : {capture_map5095-current_map5095:+.10f}")
    print("\nBootstrap henüz başlatılmadı.")
else:
    STATS_PATH = ARCHIVE_CAPTURE / "image_level_bootstrap_inputs.npz"
    assert STATS_PATH.is_file(), STATS_PATH
    capture_map50 = current_map50
    capture_map5095 = current_map5095


## 6. Recompute 5,000 image-level AP bootstrap replicates
Seed and resampling logic are preserved. This cell writes into the new output folder. Undefined class AP replicates are omitted for that class; seven-class mAP requires all seven classes.

In [ ]:
import numpy as np
import pandas as pd
import json
import time
from pathlib import Path
from tqdm.auto import tqdm
from ultralytics.utils.metrics import ap_per_class

# Kaydedilmiş girdileri oku.
STATS_PATH = Path(STATS_PATH)
BOOT_DIR = RUN_ROOT / "bootstrap_5000"
BOOT_DIR.mkdir(parents=True, exist_ok=True)

with np.load(STATS_PATH, allow_pickle=False) as archive:
    names = archive["image_names"].tolist()
    records = [
        {
            key: archive[f"{key}_{i:04d}"].copy()
            for key in ["tp", "conf", "pred_cls", "target_cls"]
        }
        for i in range(len(names))
    ]

N_IMAGES = len(records)
N_CLASSES = 7
N_BOOT = 5000
BOOT_SEED = 20261103

assert N_IMAGES == 315

def calculate_ap(indices):
    """Dönen matris: 7 sınıf × 10 IoU eşiği."""
    selected = [records[int(i)] for i in indices]

    tp = np.concatenate([r["tp"] for r in selected], axis=0)
    conf = np.concatenate([r["conf"] for r in selected])
    pred_cls = np.concatenate([r["pred_cls"] for r in selected])
    target_cls = np.concatenate([r["target_cls"] for r in selected])

    result_ap = np.full((N_CLASSES, 10), np.nan)

    if len(target_cls) == 0:
        return result_ap

    result = ap_per_class(
        tp, conf, pred_cls, target_cls,
        plot=False
    )

    # Ultralytics ap_per_class çıktısı:
    # tp, fp, precision, recall, f1, ap, unique_classes, ...
    ap_values = np.asarray(result[5])
    class_ids = np.asarray(result[6], dtype=int)

    assert ap_values.shape == (len(class_ids), 10)
    assert np.all((class_ids >= 0) & (class_ids < N_CLASSES))
    result_ap[class_ids] = ap_values

    return result_ap


# 1. Nokta tahminlerini yeniden üret.
point_ap = calculate_ap(np.arange(N_IMAGES))
assert np.isfinite(point_ap).all()

point_map50 = float(point_ap[:, 0].mean())
point_map5095 = float(point_ap.mean())

print("Kaydedilen girdilerden yeniden hesaplanan:")
print(f"mAP50    : {point_map50:.10f}")
print(f"mAP50–95 : {point_map5095:.10f}")

assert np.isclose(
    point_map50, capture_map50, atol=1e-8, rtol=0
), "mAP50 eşleşmedi; bootstrap başlatılmadı."

assert np.isclose(
    point_map5095, capture_map5095, atol=1e-8, rtol=0
), "mAP50–95 eşleşmedi; bootstrap başlatılmadı."

print("Nokta tahminleri doğrulandı.\n")


# 2. Önceki ilerleme varsa yükle.
CHECKPOINT = BOOT_DIR / "bootstrap_progress.npz"
input_hash = sha256_file(STATS_PATH)

boot_ap = np.full((N_BOOT, N_CLASSES, 10), np.nan)
completed = 0

if CHECKPOINT.exists():
    with np.load(CHECKPOINT, allow_pickle=False) as saved:
        assert str(saved["input_sha256"].item()) == input_hash
        assert int(saved["seed"].item()) == BOOT_SEED
        assert saved["ap"].shape == boot_ap.shape
        boot_ap = saved["ap"].copy()
        completed = int(saved["completed"].item())

    print(f"Önceki kayıt bulundu: {completed}/{N_BOOT} tekrar.")


def save_progress(done):
    temp_path = BOOT_DIR / "bootstrap_progress.tmp.npz"
    np.savez_compressed(
        temp_path,
        ap=boot_ap,
        completed=np.array(done),
        seed=np.array(BOOT_SEED),
        input_sha256=np.array(input_hash)
    )
    temp_path.replace(CHECKPOINT)


# 3. Görüntü düzeyinde bootstrap.
start_time = time.time()

for b in tqdm(
    range(completed, N_BOOT),
    initial=completed,
    total=N_BOOT,
    desc="Görüntü düzeyinde bootstrap"
):
    # Her tekrarın örneklemesi kesintiden bağımsız olarak tekrarlanabilir.
    rng = np.random.default_rng(
        np.random.SeedSequence([BOOT_SEED, b])
    )
    sampled_indices = rng.integers(0, N_IMAGES, size=N_IMAGES)

    # Aynı görüntü birden fazla seçilirse verileri de aynı sayıda eklenir.
    boot_ap[b] = calculate_ap(sampled_indices)

    if (b + 1) % 100 == 0 or b + 1 == N_BOOT:
        save_progress(b + 1)


# 4. Güven aralıkları.
# Referans örneği olmayan sınıfın AP'si o tekrarda NaN bırakılır.
# Yedi sınıflı mAP için yalnızca tüm sınıfların bulunduğu tekrarlar kullanılır.
all_classes_present = np.isfinite(boot_ap).all(axis=(1, 2))

map50_distribution = np.full(N_BOOT, np.nan)
map5095_distribution = np.full(N_BOOT, np.nan)

map50_distribution[all_classes_present] = (
    boot_ap[all_classes_present, :, 0].mean(axis=1)
)
map5095_distribution[all_classes_present] = (
    boot_ap[all_classes_present].mean(axis=(1, 2))
)

rows = []

def add_result(metric, class_name, estimate, distribution):
    valid = distribution[np.isfinite(distribution)]
    assert len(valid) > 0, f"Geçerli tekrar yok: {metric}, {class_name}"

    low, high = np.percentile(valid, [2.5, 97.5])
    rows.append({
        "metric": metric,
        "class_name": class_name,
        "point_estimate": float(estimate),
        "ci95_lower": float(low),
        "ci95_upper": float(high),
        "valid_replicates": len(valid),
        "undefined_replicates": N_BOOT - len(valid),
    })

add_result("mAP50", "All classes", point_map50, map50_distribution)
add_result("mAP50-95", "All classes", point_map5095, map5095_distribution)

class_names = [
    "Missing", "Dental Crown", "Root Canal", "Caries",
    "Broken Down", "Wisdom Teeth", "Healthy"
]

for c, name in enumerate(class_names):
    add_result(
        "AP50", name,
        point_ap[c, 0],
        boot_ap[:, c, 0]
    )
    add_result(
        "AP50-95", name,
        point_ap[c].mean(),
        boot_ap[:, c, :].mean(axis=1)
    )

ci_table = pd.DataFrame(rows)
ci_table.to_csv(
    BOOT_DIR / "AP_mAP_bootstrap_95CI.csv",
    index=False
)

metadata = {
    "method": "image-level nonparametric percentile bootstrap",
    "replicates": N_BOOT,
    "images_per_replicate": N_IMAGES,
    "seed": BOOT_SEED,
    "input_sha256": input_hash,
    "ultralytics": ultralytics.__version__,
    "confidence_interval_percentiles": [2.5, 97.5],
    "class_absence_rule": (
        "Class AP is undefined when no reference instance of that "
        "class is sampled; excluded from that class's percentile CI."
    ),
    "map_rule": (
        "Seven-class mAP is calculated only for replicates "
        "containing reference instances of all seven classes."
    ),
    "valid_map_replicates": int(all_classes_present.sum()),
}

(BOOT_DIR / "bootstrap_method.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

print("\nBOOTSTRAP TAMAMLANDI")
print("Bu çalıştırmanın süresi (dakika):",
      round((time.time() - start_time) / 60, 2))
display(ci_table.round(6))
print("\nSonuç klasörü:", BOOT_DIR)

## 7. Load verified normalized boxes (or optionally capture again)
Boxes are in normalized xyxy coordinates. The legacy validator-exported prediction TXT files are not used.

In [ ]:
if RECAPTURE:
    import json
    import uuid
    import numpy as np
    from datetime import datetime, timezone
    from ultralytics.models.rtdetr.val import RTDETRValidator
    
    # Her çalıştırma ayrı klasöre kaydedilir.
    capture_name = (
        "fp_fn_coordinate_capture_"
        + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
        + "_" + uuid.uuid4().hex[:6]
    )
    FPFN_DIR = RUN_ROOT / capture_name
    FPFN_DIR.mkdir(parents=True, exist_ok=False)
    
    direct_records = {}
    
    def to_numpy(tensor):
        return tensor.detach().cpu().numpy().copy()
    
    
    class DirectBoxCaptureValidator(RTDETRValidator):
        def update_metrics(self, preds, batch):
            for si, pred in enumerate(preds):
                prepared = self._prepare_batch(si, batch)
                prediction = self._prepare_pred(pred)
    
                name = Path(prepared["im_file"]).name
                assert name not in direct_records, (
                    f"Görüntü birden fazla kez işlendi: {name}"
                )
    
                input_h, input_w = map(int, prepared["imgsz"])
                divisor = np.array(
                    [input_w, input_h, input_w, input_h],
                    dtype=np.float32
                )
    
                # Hem referans hem tahmin kutuları aynı giriş
                # koordinat sisteminden normalize edilir.
                direct_records[name] = {
                    "gt_cls": to_numpy(prepared["cls"]).astype(int),
                    "gt_boxes": (
                        to_numpy(prepared["bboxes"]).reshape(-1, 4)
                        / divisor
                    ),
                    "pred_cls": to_numpy(prediction["cls"]).astype(int),
                    "pred_boxes": (
                        to_numpy(prediction["bboxes"]).reshape(-1, 4)
                        / divisor
                    ),
                    "confidence": to_numpy(prediction["conf"]),
                    "input_shape": np.array([input_h, input_w]),
                }
    
            # Standart değerlendirme işlemi korunur.
            super().update_metrics(preds, batch)
    
    
    print("Test çıkarımı ve koordinat kaydı başlıyor...")
    
    direct_metrics = model.val(
        validator=DirectBoxCaptureValidator,
        data=str(DATA_YAML),
        split="test",
        imgsz=640,
        batch=2,
        conf=0.001,
        iou=0.70,
        max_det=300,
        device=0,
        workers=2,
        half=False,
        plots=False,
        save_txt=False,
        save_json=False,
        project=str(RUN_ROOT),
        name=capture_name,
        exist_ok=True,
        verbose=False,
    )
    
    assert len(direct_records) == 315
    assert set(direct_records) == set(test_names)
    
    # Kayıtları tam hassasiyetle sakla.
    ordered_names = sorted(direct_records)
    payload = {"image_names": np.asarray(ordered_names)}
    
    for i, name in enumerate(ordered_names):
        for key, value in direct_records[name].items():
            payload[f"{key}_{i:04d}"] = value
    
    DIRECT_BOXES_PATH = FPFN_DIR / "direct_normalized_boxes.npz"
    np.savez_compressed(DIRECT_BOXES_PATH, **payload)
    
    # Referans kutularını temiz YOLO etiketleriyle karşılaştır.
    max_coordinate_difference = 0.0
    class_counts = np.zeros(7, dtype=int)
    background_count = 0
    
    for name in ordered_names:
        label_path = LABEL_DIR / f"{Path(name).stem}.txt"
        rows = [
            [float(v) for v in line.split()]
            for line in label_path.read_text().splitlines()
            if line.strip()
        ]
        labels = np.asarray(rows, dtype=np.float64).reshape(-1, 5)
        rec = direct_records[name]
    
        expected_classes = labels[:, 0].astype(int)
        xywh = labels[:, 1:5]
    
        expected_boxes = np.column_stack([
            xywh[:, 0] - xywh[:, 2] / 2,
            xywh[:, 1] - xywh[:, 3] / 2,
            xywh[:, 0] + xywh[:, 2] / 2,
            xywh[:, 1] + xywh[:, 3] / 2,
        ])
    
        assert np.array_equal(rec["gt_cls"], expected_classes), (
            f"Referans sınıf sırası farklı: {name}"
        )
        assert rec["gt_boxes"].shape == expected_boxes.shape
    
        if len(labels):
            difference = float(
                np.max(np.abs(rec["gt_boxes"] - expected_boxes))
            )
            max_coordinate_difference = max(
                max_coordinate_difference, difference
            )
            assert difference < 1e-5, (
                f"Referans koordinat uyuşmazlığı: {name}, fark={difference}"
            )
        else:
            background_count += 1
    
        class_counts += np.bincount(expected_classes, minlength=7)
    
        assert np.isfinite(rec["pred_boxes"]).all()
        assert np.isfinite(rec["confidence"]).all()
    
    assert class_counts.tolist() == [411, 282, 305, 225, 130, 132, 34]
    assert background_count == 4
    
    # Önceki bootstrap girdileriyle doğrulanan nokta tahminleri
    previous_map50 = 0.6755318157587847
    previous_map5095 = 0.29389012321773966
    
    current_map50 = float(direct_metrics.box.map50)
    current_map5095 = float(direct_metrics.box.map)
    
    metrics_match = bool(np.allclose(
        [current_map50, current_map5095],
        [previous_map50, previous_map5095],
        atol=1e-6,
        rtol=0
    ))
    
    summary = {
        "images": len(direct_records),
        "annotations": int(class_counts.sum()),
        "annotations_per_class": class_counts.tolist(),
        "background_images": background_count,
        "maximum_reference_coordinate_difference": max_coordinate_difference,
        "map50": current_map50,
        "map50_95": current_map5095,
        "difference_map50": current_map50 - previous_map50,
        "difference_map50_95": current_map5095 - previous_map5095,
        "previous_metrics_match_atol_1e_6": metrics_match,
        "coordinate_source": "RTDETRValidator prepared input-space boxes",
        "normalization": "xyxy divided by input width, height, width, height",
    }
    
    (FPFN_DIR / "coordinate_validation.json").write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8"
    )
    
    print("\nKOORDİNAT VE SONUÇ KONTROLÜ")
    print(json.dumps(summary, indent=2))
    print("\nKaydedilen kutular:", DIRECT_BOXES_PATH)
    print("Referans koordinat kontrolü: BAŞARILI")
    print("Önceki mAP sonuçlarıyla uyum:", metrics_match)
    print("\nHenüz FP/FN örneği seçilmedi veya şekil oluşturulmadı.")
else:
    FPFN_DIR = RUN_ROOT / "fp_fn"
    FPFN_DIR.mkdir(parents=True, exist_ok=False)
    DIRECT_BOXES_PATH = ARCHIVE_FIG / "direct_normalized_boxes.npz"
    with np.load(DIRECT_BOXES_PATH, allow_pickle=False) as a:
        direct_records = {
            str(name): {key: a[f"{key}_{i:04d}"].copy() for key in
                ["gt_cls", "gt_boxes", "pred_cls", "pred_boxes", "confidence", "input_shape"]}
            for i, name in enumerate(a["image_names"].tolist())
        }
    assert set(direct_records) == set(test_names)
    assert sum(len(r["gt_cls"]) for r in direct_records.values()) == 1519
    print("Loaded verified image records:", len(direct_records))


## 8. Class-aware, confidence-ordered matching for the illustration
These diagnostic counts do not replace the archived fixed-threshold manuscript table. Observed differences remain documented in README.

In [ ]:
import numpy as np
import pandas as pd
import json

CONF_THRESHOLD = 0.55
MATCH_IOU = 0.50
CLASS_NAMES = [
    "Missing", "Dental Crown", "Root Canal", "Caries",
    "Broken Down", "Wisdom Teeth", "Healthy"
]

def pairwise_iou(a, b):
    a = np.asarray(a, dtype=float).reshape(-1, 4)
    b = np.asarray(b, dtype=float).reshape(-1, 4)

    left_top = np.maximum(a[:, None, :2], b[None, :, :2])
    right_bottom = np.minimum(a[:, None, 2:], b[None, :, 2:])
    intersection = np.maximum(right_bottom - left_top, 0).prod(axis=2)

    area_a = np.maximum(a[:, 2:] - a[:, :2], 0).prod(axis=1)
    area_b = np.maximum(b[:, 2:] - b[:, :2], 0).prod(axis=1)
    union = area_a[:, None] + area_b[None, :] - intersection
    return intersection / np.maximum(union, 1e-12)


error_rows = []
count_rows = []
fpfn_records = {}

for name in sorted(direct_records):
    original = direct_records[name]
    keep = original["confidence"] >= CONF_THRESHOLD

    gt_cls = original["gt_cls"]
    gt_boxes = original["gt_boxes"]
    pred_cls = original["pred_cls"][keep]
    pred_boxes = original["pred_boxes"][keep]
    confidence = original["confidence"][keep]

    overlaps = pairwise_iou(pred_boxes, gt_boxes)
    matched_gt = set()
    matched_pred = set()

    # Önce yüksek güvenli tahminler; eşitliklerde mevcut sıra korunur.
    for pi in np.argsort(-confidence, kind="stable"):
        candidates = [
            gi for gi in range(len(gt_cls))
            if gi not in matched_gt
            and gt_cls[gi] == pred_cls[pi]
            and overlaps[pi, gi] >= MATCH_IOU
        ]
        if candidates:
            gi = max(candidates, key=lambda g: overlaps[pi, g])
            matched_gt.add(int(gi))
            matched_pred.add(int(pi))

    fpfn_records[name] = {
        "gt_cls": gt_cls,
        "gt_boxes": gt_boxes,
        "pred_cls": pred_cls,
        "pred_boxes": pred_boxes,
        "confidence": confidence,
        "matched_gt": matched_gt,
        "matched_pred": matched_pred,
    }

    for cls in range(7):
        tp = sum(pred_cls[pi] == cls for pi in matched_pred)
        n_pred = int(np.sum(pred_cls == cls))
        n_gt = int(np.sum(gt_cls == cls))
        count_rows.append({
            "image_name": name,
            "class_id": cls,
            "TP": int(tp),
            "FP": n_pred - int(tp),
            "FN": n_gt - int(tp),
            "reference_boxes": n_gt,
            "predictions": n_pred,
        })

    # Referansla eşleşmeyen tahminler: FP
    for pi in range(len(pred_cls)):
        if pi in matched_pred:
            continue
        cls = int(pred_cls[pi])
        same_class = np.flatnonzero(gt_cls == cls)
        box = pred_boxes[pi]

        error_rows.append({
            "image_name": name,
            "error_type": "FP",
            "class_id": cls,
            "class_name": CLASS_NAMES[cls],
            "box_index": pi,
            "confidence": float(confidence[pi]),
            "max_same_class_iou": (
                float(overlaps[pi, same_class].max())
                if len(same_class) else 0.0
            ),
            "box_area": float(np.maximum(box[2:] - box[:2], 0).prod()),
        })

    # Tahminle eşleşmeyen referans kutuları: FN
    for gi in range(len(gt_cls)):
        if gi in matched_gt:
            continue
        cls = int(gt_cls[gi])
        same_class = np.flatnonzero(pred_cls == cls)
        box = gt_boxes[gi]

        error_rows.append({
            "image_name": name,
            "error_type": "FN",
            "class_id": cls,
            "class_name": CLASS_NAMES[cls],
            "box_index": gi,
            "confidence": np.nan,
            "max_same_class_iou": (
                float(overlaps[same_class, gi].max())
                if len(same_class) else 0.0
            ),
            "box_area": float(np.maximum(box[2:] - box[:2], 0).prod()),
        })

errors = pd.DataFrame(error_rows)
counts = pd.DataFrame(count_rows)

summary = (
    counts.groupby("class_id")[
        ["reference_boxes", "predictions", "TP", "FP", "FN"]
    ]
    .sum()
    .reindex(range(7))
)
summary.insert(0, "class_name", CLASS_NAMES)

assert int(summary["reference_boxes"].sum()) == 1519
assert (summary["TP"] + summary["FN"] == summary["reference_boxes"]).all()
assert (summary["TP"] + summary["FP"] == summary["predictions"]).all()

# Örnek adaylarının tamamını sakla.
errors.to_csv(FPFN_DIR / "fp_fn_candidates.csv", index=False)
counts.to_csv(FPFN_DIR / "image_class_matching_counts.csv", index=False)
summary.to_csv(FPFN_DIR / "matching_summary.csv", index_label="class_id")

method = {
    "confidence_threshold": CONF_THRESHOLD,
    "matching_iou": MATCH_IOU,
    "matching": (
        "Class-aware, one-to-one greedy matching in descending prediction "
        "confidence order; highest-IoU unmatched eligible reference selected."
    ),
    "purpose": "Candidate identification for illustrative FP/FN figures",
    "reference": "Dataset annotations; no independent clinical adjudication",
}
(FPFN_DIR / "fp_fn_matching_method.json").write_text(
    json.dumps(method, indent=2), encoding="utf-8"
)

print("SINIF BAZINDA EŞLEŞTİRME")
display(summary)

print("\nFP/FN ADAY SAYILARI")
display(pd.crosstab(errors["class_name"], errors["error_type"]))

print("\nEN YÜKSEK GÜVENLİ 10 FP ADAYI")
display(
    errors[errors["error_type"] == "FP"]
    .sort_values("confidence", ascending=False)
    .head(10)[[
        "image_name", "class_name", "confidence", "max_same_class_iou"
    ]]
)

print("\nKayıt klasörü:", FPFN_DIR)
print("Henüz nihai örnek seçilmedi veya şekil oluşturulmadı.")

## 9. Restore the exact six selected examples and render Figure S1
Selection was illustrative, not a frequency estimate. Archived row indices are checked against class, error status, confidence and overlap. On a changed inference result, the notebook stops instead of silently substituting examples.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from PIL import Image
final_examples = pd.read_csv(SELECTION_PATH)
assert final_examples["panel"].tolist() == list("ABCDEF")
assert final_examples["image_name"].tolist() == ["00001024.jpg", "0000652.jpg", "ibn128.jpg", "00001017.jpg", "0000683.jpg", "ibn364.jpg"]
for _, row in final_examples.iterrows():
    match = errors[(errors.image_name == row.image_name) &
                   (errors.error_type == row.error_type) &
                   (errors.class_id == row.class_id) &
                   (errors.box_index == row.box_index)]
    assert len(match) == 1, f"Selected error changed: {row.panel}"
    observed = match.iloc[0]
    assert np.isclose(observed.max_same_class_iou, row.max_same_class_iou, atol=1e-6, rtol=0)
    if row.error_type == "FP":
        assert np.isclose(observed.confidence, row.confidence, atol=1e-6, rtol=0)
GT_COLOR, PRED_COLOR = "#00D5FF", "#FFB000"
def draw_box(ax, box, w, h, color, lw=1.2, style="-", alpha=0.65):
    x1,y1,x2,y2 = np.asarray(box) * [w,h,w,h]
    ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,fill=False,
                          edgecolor=color,linewidth=lw,linestyle=style,alpha=alpha))
    return ((x1+x2)/2,(y1+y2)/2)
fig, axes = plt.subplots(3,2,figsize=(16,12))
for ax, (_,row) in zip(axes.flat, final_examples.iterrows()):
    r=fpfn_records[row.image_name]; cls=int(row.class_id); idx=int(row.box_index)
    with Image.open(IMAGE_DIR / row.image_name) as im:
        im=im.convert("RGB"); w,h=im.size; ax.imshow(im)
    for gi in np.flatnonzero(r["gt_cls"] == cls):
        draw_box(ax,r["gt_boxes"][gi],w,h,GT_COLOR)
    for pi in np.flatnonzero(r["pred_cls"] == cls):
        draw_box(ax,r["pred_boxes"][pi],w,h,PRED_COLOR,style="--")
    fp=row.error_type == "FP"
    center=draw_box(ax,r["pred_boxes" if fp else "gt_boxes"][idx],w,h,
                    PRED_COLOR if fp else GT_COLOR,2.6,"--" if fp else "-",1)
    ax.annotate(row.error_type,xy=center,xytext=(0.04,0.93),textcoords="axes fraction",
                color="white",fontsize=11,bbox=dict(facecolor="black",edgecolor="none",alpha=0.8),
                arrowprops=dict(arrowstyle="->",color="white",linewidth=1.4))
    ax.set_title(f"{row.panel}. {row.error_type} — {row.class_name}",loc="left",fontsize=11)
    ax.set_xlim(0,w); ax.set_ylim(h,0); ax.axis("off")
fig.legend(handles=[Line2D([0],[0],color=GT_COLOR,lw=2,label="Reference annotation"),
                    Line2D([0],[0],color=PRED_COLOR,lw=2,linestyle="--",label="Model prediction")],
           loc="lower center",ncol=2,frameon=False)
fig.tight_layout(rect=[0,0.045,1,1])
fig.savefig(FPFN_DIR / "Supplementary_Figure_S1_300dpi.png",dpi=300,bbox_inches="tight",facecolor="white")
fig.savefig(FPFN_DIR / "Supplementary_Figure_S1.pdf",bbox_inches="tight",facecolor="white")
final_examples.to_csv(FPFN_DIR / "final_selected_examples.csv",index=False)
plt.show(); plt.close(fig)
print("Saved in", FPFN_DIR)

## 10. Collect the archived evidence into a local ZIP for review
This does not upload or publish anything. Original image files and model weights are not bundled. The figure itself contains source radiographs: check the applicable redistribution terms before public release. Add this cleaned notebook and README to the resulting evidence folder before publishing.

In [ ]:
import hashlib, zipfile
EVIDENCE_DIR = RUN_ROOT / "revision2_archived_evidence"
EVIDENCE_DIR.mkdir(exist_ok=False)
required = {
 "bootstrap/image_level_bootstrap_inputs.npz": ARCHIVE_CAPTURE / "image_level_bootstrap_inputs.npz",
 "bootstrap/AP_mAP_bootstrap_95CI.csv": ARCHIVE_CAPTURE / "bootstrap_5000/AP_mAP_bootstrap_95CI.csv",
 "bootstrap/bootstrap_method.json": ARCHIVE_CAPTURE / "bootstrap_5000/bootstrap_method.json",
 "figures/direct_normalized_boxes.npz": ARCHIVE_FIG / "direct_normalized_boxes.npz",
 "figures/final_selected_examples.csv": SELECTION_PATH,
 "figures/fp_fn_matching_method.json": ARCHIVE_FIG / "fp_fn_matching_method.json",
 "figures/coordinate_validation.json": ARCHIVE_FIG / "coordinate_validation.json",
 "figures/Supplementary_Figure_S1_300dpi.png": ARCHIVE_FIG / "Supplementary_Figure_S1_300dpi.png",
 "figures/Supplementary_Figure_S1.pdf": ARCHIVE_FIG / "Supplementary_Figure_S1.pdf",
}
for relative, source in required.items():
    assert source.is_file(), source
for relative, source in required.items():
    dest=EVIDENCE_DIR / relative; dest.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(source,dest)
manifest={str(p.relative_to(EVIDENCE_DIR)):sha256_file(p) for p in sorted(EVIDENCE_DIR.rglob("*")) if p.is_file()}
(EVIDENCE_DIR / "SHA256.json").write_text(json.dumps(manifest,indent=2))
zip_path=Path(shutil.make_archive(str(EVIDENCE_DIR),"zip",EVIDENCE_DIR))
print("Evidence ZIP (not yet published):",zip_path)